# Dimensions and DimensionSets

## Dimensions

[Dimension](api.dimensions.html#flodym.Dimension) objects are what flodym datasets are differentiated by.

They are defined by a name, index letter, and their items.
The index letters must have length one, i.e. only be one letter.

Optionally, a data type can be given. If doing so, it is ensured that all items have this data type.
When loading items from file, it can be useful to check that for example years are integers and not converted to strings.

In [1]:
import flodym as fd

regions = fd.Dimension(name="Region", letter="r", items=["EU", "US", "MEX"])


Never give Dimension objects with different item sets the same name or letter, even though it is tempting!

In [2]:
# historic and future time
time = fd.Dimension(name="Time", letter="t", items=["2010", "2020", "2030"])
# Historic time - a subset of time

# WRONG! (same letter t)
historic_time = fd.Dimension(name="Historic Time", letter="t", items=["2010", "2020"])

# Correct
historic_time = fd.Dimension(name="Historic Time", letter="h", items=["2010", "2020"])

## DimensionSets

Multiple dimensions are grouped in a DimensionSet object:

In [3]:
dims = fd.DimensionSet(
    dim_list=[
        fd.Dimension(name="Region", letter="r", items=["EU", "US", "MEX"]),
        fd.Dimension(name="Product", letter="p", items=["A", "B"]),
        fd.Dimension(name="Time", letter="t", items=[2020]),
    ]
)

The MFASystem class has a dims attribute, which is a DimensionSet object containing all dimensions ever used in the system.

Each FlodymArray object also has a dims attribute, with a subset of dimensions that this array is defined over.

Subsets and single dimensions can be extracted by indexing. A single index in square brackets returns a `Dimension`.
Several indexes in the square brackets, or a tuple (even with a single item inside), return a `DimensionSet`.

In [4]:
item_by_letter = dims["r"]
print(f"Item by letter | type: {type(item_by_letter)}, name: {item_by_letter.name}")

item_by_name = dims["Region"]
print(f"Item by name | type: {type(item_by_name)}, name: {item_by_name.name}")

subset = dims["r", "p"]
print(f"Subset | type: {type(subset)}, names: {subset.names}")

subset_size_1_tuple = dims[("r",)]
print(f"Size 1 subset | type: {type(subset_size_1_tuple)}, names: {subset_size_1_tuple.names}")

Item by letter | type: <class 'flodym.dimensions.Dimension'>, name: Region
Item by name | type: <class 'flodym.dimensions.Dimension'>, name: Region
Subset | type: <class 'flodym.dimensions.DimensionSet'>, names: ('Region', 'Product')
Size 1 subset | type: <class 'flodym.dimensions.DimensionSet'>, names: ('Region',)


DimensionSet objects have several attributes (like `names` or `letters`) and a range of methods to modify or combine them.
For a full list and documentation, refer to the API Reference.
Here we list the few most important ones.

In [5]:

# Transform single Dimension to DimensionSet
dim_as_set = dims["r"].as_dimset()
print(f"Dimension as DimensionSet | type: {type(dim_as_set)}, names: {dim_as_set.names}")

# Set union of two DimensionSets
combined_dims = dims["r", "p"] | dims["p", "t"]
print(f"Combined DimensionSet | type: {type(combined_dims)}, names: {combined_dims.names}")

# Set intersection of two DimensionSets
intersected_dims = dims["r", "p"] & dims["p", "t"]
print(f"Intersected DimensionSet | type: {type(intersected_dims)}, names: {intersected_dims.names}")

# Remove dimensions from DimensionSet
reduced_dims = dims - dims["t"]
print(f"Reduced DimensionSet | type: {type(reduced_dims)}, names: {reduced_dims.names}")

Dimension as DimensionSet | type: <class 'flodym.dimensions.DimensionSet'>, names: ('Region',)
Combined DimensionSet | type: <class 'flodym.dimensions.DimensionSet'>, names: ('Region', 'Product', 'Time')
Intersected DimensionSet | type: <class 'flodym.dimensions.DimensionSet'>, names: ('Product',)
Reduced DimensionSet | type: <class 'flodym.dimensions.DimensionSet'>, names: ('Region', 'Product')


## Sub-dimensions

A dimension can be a finer resolution of another dimension, called its parent.
Each item is assigned to exactly one item of the parent.
Typical examples are countries assigned to world regions, or building types split into sub-types.

A mapping from items to parent items can be given directly:

In [ ]:
world_regions = fd.Dimension(name="World region", letter="w", items=["EUR", "USA"])
countries = fd.Dimension.from_mapping(
    name="Country",
    letter="c",
    parent=world_regions,
    mapping={"DEU": "EUR", "FRA": "EUR", "USA": "USA"},
)
print(countries)

Alternatively, some items of the parent can be split into the items of other dimensions.
Items which are not split are kept. Several levels of splitting are possible.

In [ ]:
building_types = fd.Dimension(name="Building type", letter="b", items=["Com", "Res"])
residential_types = fd.Dimension(name="Residential type", letter="y", items=["Sin", "Mul"])
building_sub_types = building_types.split(
    name="Building sub-type", letter="s", splitting={"Res": residential_types}
)
print(building_sub_types.items)

Parent and sub-dimension have different letters, so arrays can be defined over either of them.

Arrays can be aggregated to a coarser dimension with `aggregate_to()` and disaggregated to a finer
one with `disaggregate_to()`.
Whether weights are needed depends on whether the array is extensive or intensive (see the
HOWTO on arrays):

- Extensive quantities such as flows are summed when aggregating, but need shares or weights when
  splitting.
- Intensive quantities such as material intensities are copied when splitting, but need weights
  for the weighted mean when aggregating.

Weights are normalized within each parent item, so they can be shares or absolute values, e.g.
production volumes.
Using one of the methods in a way that does not fit the `extensive` flag of the array is reported.

In [ ]:
import numpy as np

production = fd.FlodymArray(
    dims=building_types.as_dimset(), values=np.array([1.0, 10.0]), name="production", extensive=True
)
shares = fd.FlodymArray(
    dims=building_sub_types.as_dimset(), values=np.array([1.0, 0.8, 0.2]), extensive=False
)

production_by_sub_type = production.disaggregate_to(building_sub_types, weights=shares)
print(production_by_sub_type.values)

material_intensity = fd.FlodymArray(
    dims=building_sub_types.as_dimset(), values=np.array([5.0, 10.0, 20.0]), extensive=False
)
mean_intensity = material_intensity.aggregate_to(building_types, weights=production_by_sub_type)
print(mean_intensity.values)

Operations between arrays over related dimensions treat them like other dimensions, which only
one operand has: Reducing sums, expanding copies.

- `+`, `-`, `minimum()`, `maximum()`, `sum_to()` and assignments such as `coarse[...] = fine`
  aggregate to the coarser dimension by summation.
  For intensive arrays, `+`, `-`, `minimum()` and `maximum()` instead disaggregate to the finer
  dimension by copying.
- `*`, `/` and `cast_to()` disaggregate to the finer dimension by copying values.
- Assignments such as `fine[...] = coarse` copy values for intensive arrays, and raise an error
  otherwise.

Splitting extensive quantities and aggregating intensive quantities is never done
automatically; use the methods above with weights.

In [ ]:
material_demand = production_by_sub_type * material_intensity
print(material_demand.dims.letters, material_demand.values)

material_demand_by_type = fd.FlodymArray(dims=building_types.as_dimset(), extensive=True)
material_demand_by_type[...] = material_demand
print(material_demand_by_type.values)

For a split dimension, a split item of the parent addresses its sub-items.
The result is defined over the dimension the item was split into.

In [ ]:
residential_intensity = material_intensity["Res"]
print(residential_intensity.dims.letters, residential_intensity.values)

When setting values, any item of the parent or a further ancestor addresses all items belonging
to it, also for dimensions created from a mapping.
This is not possible when getting values, as no dimension exists for the result.

In [ ]:
material_intensity[{"s": "Res"}] = 15.0
print(material_intensity.values)

### Several levels of nesting

Splitting can be repeated, so a dimension can have a parent, which has a parent itself, and so on.
Such a dimension is built in three steps:

1. Create the flat dimensions of each level, here `coarse`, `medium` and `fine`.
2. Split each level by the next finer one: `coarse_and_medium` splits `"Build"` into the items of
   `medium`, and `medium_and_fine` splits `"Res"` into the items of `fine`.
3. Combine both splits with `Dimension.combine_splits()` into `coarse_and_medium_and_fine`.

The combined dimension has `coarse_and_medium` as parent, which has `coarse` as parent.
So arrays over it can be aggregated to each level, combined with arrays over each level, and
sliced by the items of each level.
Getting values by an item returns an array over the dimension the item was split into:
`"Build|Res"` gives an array over `fine`, and `"Build"` an array over `medium_and_fine`.

In [ ]:
coarse = fd.Dimension(name="Construction type", letter="i", items=["Infr", "Build"])
medium = fd.Dimension(name="Building type", letter="b", items=["Com", "Res"])
fine = fd.Dimension(name="Residential type", letter="y", items=["Sin", "Mul"])

coarse_and_medium = coarse.split(name="Level 1", letter="l", splitting={"Build": medium})
medium_and_fine = medium.split(name="Building sub-type", letter="s", splitting={"Res": fine})
coarse_and_medium_and_fine = fd.Dimension.combine_splits(
    name="Level 2", letter="m", upper=coarse_and_medium, lower=medium_and_fine
)
print(coarse_and_medium_and_fine.items)

stock = fd.FlodymArray(
    dims=coarse_and_medium_and_fine.as_dimset(),
    values=np.array([1.0, 2.0, 3.0, 4.0]),
    extensive=True,
)
print(stock["Build"].dims.letters, stock["Build"].values)
print(stock["Build|Res"].dims.letters, stock["Build|Res"].values)
print(stock.aggregate_to(coarse_and_medium).values)

When reading dimensions with a data reader, sub-dimensions are defined in the
`DimensionDefinition` by the letter of their `parent`.
The dimension file then has two columns: the items, and the parent item of each item.
A split dimension is defined by `splitting`, which gives the dimension each parent item is split
into. It is not read from a file, but created from the other dimensions.

In [ ]:
dimension_definitions = [
    fd.DimensionDefinition(name="World region", letter="w", dtype=str),
    fd.DimensionDefinition(name="Country", letter="c", dtype=str, parent="w"),
    fd.DimensionDefinition(name="Building type", letter="b", dtype=str),
    fd.DimensionDefinition(name="Residential type", letter="y", dtype=str),
    fd.DimensionDefinition(
        name="Building sub-type", letter="s", dtype=str, parent="b", splitting={"Res": "y"}
    ),
]